# 📓 Lección 3: Distribución de Probabilidad
**Proyecto:** Análisis estadístico sobre hábitos saludables en jóvenes universitarios  
**Módulo 6 – Inferencia Estadística | Alkemy**

---

## 🎯 Objetivo
Aplicar distribuciones adecuadas según el tipo de variable y contexto.

## Setup – Cargar Dataset

In [ ]:
import random
import math
from collections import Counter

random.seed(42)

carreras = ["Ingeniería","Medicina","Derecho","Psicología","Administración","Diseño","Comunicación","Educación"]
generos = ["Masculino","Femenino","No binario"]
vive_con = ["Familia","Solo","Compañeros","Pareja"]
dieta = ["Omnívora","Vegetariana","Vegana","Otro"]
actividad_freq = ["Nunca","1-2 veces/semana","3-4 veces/semana","5+ veces/semana"]

datos = []
for i in range(1, 151):
    genero = random.choices(generos, weights=[45,50,5])[0]
    carrera = random.choice(carreras)
    edad = random.randint(18, 28)
    anio = random.randint(1, 5)
    horas_sueno = round(max(4.0, min(10.0, random.gauss(6.8, 1.1))), 1)
    comidas_dia = random.choices([1,2,3,4,5], weights=[5,15,40,30,10])[0]
    actividad = random.choices(actividad_freq, weights=[20,35,30,15])[0]
    horas_estudio = round(max(1.0, min(12.0, random.gauss(5.2, 1.8))), 1)
    consume_alcohol = random.choices(["Sí","No"], weights=[55,45])[0]
    fuma = random.choices(["Sí","No"], weights=[20,80])[0]
    estres = random.choices(["Bajo","Medio","Alto","Muy alto"], weights=[10,30,40,20])[0]
    uso_celular = round(max(1.0, min(12.0, random.gauss(5.5, 2.0))), 1)
    imc = round(max(17.0, min(38.0, random.gauss(23.5, 3.5))), 1)
    satisfaccion = random.randint(1, 10)
    vive = random.choices(vive_con, weights=[50,15,25,10])[0]
    tipo_dieta = random.choices(dieta, weights=[65,20,8,7])[0]
    trabaja = random.choices(["Sí","No"], weights=[35,65])[0]
    datos.append({
        "id": i, "edad": edad, "genero": genero, "carrera": carrera,
        "anio": anio, "horas_sueno": horas_sueno, "comidas_dia": comidas_dia,
        "actividad": actividad, "horas_estudio": horas_estudio,
        "consume_alcohol": consume_alcohol, "fuma": fuma, "estres": estres,
        "uso_celular": uso_celular, "imc": imc, "satisfaccion": satisfaccion,
        "vive_con": vive, "tipo_dieta": tipo_dieta, "trabaja": trabaja
    })

print(f"Dataset cargado: {len(datos)} registros")
print(f"Variables: {list(datos[0].keys())}")


## 1. Identificación de Distribuciones

| Variable | Distribución | Justificación |
|----------|-------------|---------------|
| Horas de sueño | **Normal** | Variable continua, distribución simétrica alrededor de la media |
| IMC | **Normal** | Variable fisiológica continua, comportamiento normal en poblaciones |
| Estrés alto (sí/no) | **Binomial** | Variable dicotómica: cada estudiante "tiene" o "no tiene" estrés alto |
| Comidas por día | **Binomial** | Número de "éxitos" (comer bien) en n intentos diarios |

In [ ]:
# === DISTRIBUCIÓN NORMAL – Horas de sueño ===

suenos = [d["horas_sueno"] for d in datos]
n = len(suenos)
media = sum(suenos) / n
varianza = sum((x - media)**2 for x in suenos) / n
std = math.sqrt(varianza)

print("=== DISTRIBUCIÓN NORMAL – Horas de sueño ===")
print(f"  N (tamaño)            : {n}")
print(f"  Media (µ)             : {media:.4f} horas")
print(f"  Desviación estándar σ : {std:.4f} horas")
print(f"  Varianza (σ²)         : {varianza:.4f}")
print(f"  Mínimo                : {min(suenos):.1f} horas")
print(f"  Máximo                : {max(suenos):.1f} horas")


In [ ]:
# Calcular P(X < 7) usando la distribución normal
# Estandarizamos: Z = (X - µ) / σ

def norm_cdf(z):
    """CDF normal estándar aproximada"""
    return (1.0 + math.erf(z / math.sqrt(2))) / 2.0

z_7 = (7 - media) / std
prob_menos_7 = norm_cdf(z_7)

print("=== CÁLCULO DE PROBABILIDADES – Normal ===")
print(f"\nP(X < 7) usando estandarización:")
print(f"  Z = (7 - {media:.4f}) / {std:.4f} = {z_7:.4f}")
print(f"  P(Z < {z_7:.4f}) = {prob_menos_7:.4f}  ({prob_menos_7*100:.1f}%)")
print()

# Regla empírica
print("Regla empírica (68-95-99.7):")
print(f"  68% de datos entre: [{media-std:.2f}, {media+std:.2f}] horas")
print(f"  95% de datos entre: [{media-2*std:.2f}, {media+2*std:.2f}] horas")
print(f"  99.7% de datos entre: [{media-3*std:.2f}, {media+3*std:.2f}] horas")


## 2. Distribución Binomial – Estrés Alto

Modelamos la variable "¿tiene estrés alto?" como una binomial con parámetros n y p.

In [ ]:
# === DISTRIBUCIÓN BINOMIAL – Estrés alto ===

from math import comb

estres_alto_count = sum(1 for d in datos if d["estres"] in ["Alto", "Muy alto"])
p_estres = estres_alto_count / len(datos)
n_grupo = 10   # tamaño del grupo analizado

print("=== DISTRIBUCIÓN BINOMIAL – Estrés Alto ===")
print(f"  Estudiantes con estrés alto en dataset : {estres_alto_count}/150")
print(f"  Probabilidad de éxito (p)              : {p_estres:.4f} ({p_estres*100:.1f}%)")
print(f"  n por grupo                            : {n_grupo}")
print()

# Calcular P(X = k) para k = 0..n
print(f"Distribución completa para n={n_grupo}, p={p_estres:.2f}:")
print(f"  {'k':>4}  {'P(X=k)':>10}  {'P(X=k)%':>10}")
print("  " + "-" * 30)
for k in range(n_grupo + 1):
    prob_k = comb(n_grupo, k) * (p_estres**k) * ((1-p_estres)**(n_grupo-k))
    marca = " ←" if k == round(n_grupo * p_estres) else ""
    print(f"  {k:>4}  {prob_k:>10.6f}  {prob_k*100:>9.2f}%{marca}")


In [ ]:
# Cálculo específico pedido: P(X = 6) de 10 estudiantes

k_especifico = 6
prob_6 = comb(n_grupo, k_especifico) * (p_estres**k_especifico) * ((1-p_estres)**(n_grupo-k_especifico))

print(f"\n=== P(X = {k_especifico}) de {n_grupo} estudiantes tengan estrés alto ===")
print(f"  Fórmula: C({n_grupo},{k_especifico}) × {p_estres:.2f}^{k_especifico} × {1-p_estres:.2f}^{n_grupo-k_especifico}")
print(f"  C({n_grupo},{k_especifico}) = {comb(n_grupo, k_especifico)}")
print(f"  {p_estres:.2f}^{k_especifico} = {p_estres**k_especifico:.6f}")
print(f"  {1-p_estres:.2f}^{n_grupo-k_especifico} = {(1-p_estres)**(n_grupo-k_especifico):.6f}")
print(f"  P(X = {k_especifico}) = {prob_6:.4f}  ({prob_6*100:.2f}%)")


## 3. Justificación de las Distribuciones

In [ ]:
# Resumen de justificación

justificacion = {
    "Normal (sueño, IMC, estudio)": [
        "Variables continuas → pueden tomar cualquier valor en un rango",
        "En poblaciones grandes se distribuyen simétricamente (campana)",
        "El TLC garantiza que la distribución muestral también sea normal"
    ],
    "Binomial (estrés alto, fuma, trabaja)": [
        "Variables dicotómicas → solo dos resultados posibles (éxito/fracaso)",
        "Cada observación es independiente",
        "La probabilidad p es constante en toda la población"
    ]
}

for dist, razones in justificacion.items():
    print(f"\n{dist}:")
    for r in razones:
        print(f"  ✓ {r}")


---
## ✅ Resumen Lección 3
- Variables continuas modeladas con distribución normal ✔️
- Variables dicotómicas modeladas con distribución binomial ✔️
- Parámetros µ y σ calculados desde el dataset ✔️
- P(X < 7) calculado mediante estandarización Z ✔️
- Distribución binomial completa calculada para n=10 ✔️

➡️ *Continuar con Lección 4: Distribución Muestral y TLC*